In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

In [2]:
# Для генерации случайных разбиений и инициализации моделей
random_seeds = [1, 42, 100, 200, 500, 1000, 2000, 3000, 4000, 5000]

In [3]:
# грузим датасет, удаляем объявления вне мкада и те, к которым внутри МКАДа не подтянулся АО
dataset = pd.read_excel('../Dataset/objects_w_geo_prepared_V0_2j.xlsx')
dataset = dataset.loc[dataset['inMKAD']==1].reset_index(drop=True)
dataset = dataset.loc[~(dataset['AO']=='НЕОПР')].reset_index(drop=True)
dataset = dataset.loc[dataset['office_class'].isin(['B', 'B+'])].reset_index(drop=True)
dataset

,Unnamed: 0,space_m2,price_per_m2,total_price,office_class,floor,maxfloor,isCU,isNonCU,isEE,...,TotalOfficeCnt_2000,classACnt_500,classACnt_1000,classACnt_2000,isMoscowCity,isSkolkovo,inMKAD,AO,inGardenRing,distanceToCenter
0,32,5.0,2400,12000,B,9,16,1,0,1,...,6,0,3,0,0,0,1,ЮЗАО,0,16425.314440
1,34,5.0,2400,12000,B+,3,15,1,0,1,...,6,0,3,0,0,0,1,ЮЗАО,0,16425.314440
2,36,5.0,2400,12000,B+,12,15,1,0,1,...,6,0,3,0,0,0,1,ЮЗАО,0,16425.314440
3,37,5.0,2400,12000,B+,4,15,1,0,1,...,6,0,3,0,0,0,1,ЮЗАО,0,16425.314440
4,39,5.0,2400,12000,B+,8,15,1,0,1,...,6,0,3,0,0,0,1,ЮЗАО,0,16425.314440
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11702,33524,16000.0,4167,66672000,B,12,16,0,1,0,...,1695,9,303,703,0,0,1,САО,0,4215.373029
11703,33574,25200.0,2917,73508400,B,1,0,0,1,0,...,228,10,27,34,0,0,1,ЮАО,0,8826.038745
11704,33577,13703.0,5473,75000000,B+,1,3,0,0,0,...,2022,28,144,684,0,0,1,ЦАО,1,1147.612056
11705,33604,9700.0,8845,85796496,B,8,12,0,1,0,...,2525,22,122,804,0,0,1,ЦАО,0,2646.298315


In [4]:
dataset.columns

Index(['Unnamed: 0', 'space_m2', 'price_per_m2', 'total_price', 'office_class',
       'floor', 'maxfloor', 'isCU', 'isNonCU', 'isEE', 'avg_latitude',
       'avg_longtitude', 'isMall', 'isOfficeCenter', 'inTTK', 'target',
       'minDistSubway', 'cafe_500', 'cafe_1000', 'cafe_2000', 'fast_food_500',
       'fast_food_1000', 'fast_food_2000', 'restaurant_500', 'restaurant_1000',
       'restaurant_2000', 'bank_500', 'bank_1000', 'bank_2000',
       'supermarket_500', 'supermarket_1000', 'supermarket_2000',
       'clothes_500', 'clothes_1000', 'clothes_2000', 'cosmetics_500',
       'cosmetics_1000', 'cosmetics_2000', 'electronics_500',
       'electronics_1000', 'electronics_2000', 'mall_500', 'mall_1000',
       'mall_2000', 'histplaces_500', 'histplaces_1000', 'histplaces_2000',
       'coords_str', 'TotalOfficeCnt_500', 'TotalOfficeCnt_1000',
       'TotalOfficeCnt_2000', 'classACnt_500', 'classACnt_1000',
       'classACnt_2000', 'isMoscowCity', 'isSkolkovo', 'inMKAD', 'AO',
     

In [5]:
### Размечаем уникальные здания
dataset['coords_str'] = [str(round(dataset.at[i, 'avg_latitude'], 6)) + '_' + str(round(dataset.at[i, 'avg_longtitude'], 6)) for i in dataset.index]
# число уникальных точек
print('Уникальных адресов:', len(set(dataset['coords_str'])))

l1 = sorted(list(set(dataset['coords_str'])))

address_to_num = {val: i for i, val in enumerate(l1)}
dataset['address_index'] = [address_to_num[i] for i in dataset['coords_str']]
print(min(dataset.address_index), max(dataset.address_index))
dataset

Уникальных адресов: 1690
0 1689


,Unnamed: 0,space_m2,price_per_m2,total_price,office_class,floor,maxfloor,isCU,isNonCU,isEE,...,classACnt_500,classACnt_1000,classACnt_2000,isMoscowCity,isSkolkovo,inMKAD,AO,inGardenRing,distanceToCenter,address_index
0,32,5.0,2400,12000,B,9,16,1,0,1,...,0,3,0,0,0,1,ЮЗАО,0,16425.314440,32
1,34,5.0,2400,12000,B+,3,15,1,0,1,...,0,3,0,0,0,1,ЮЗАО,0,16425.314440,32
2,36,5.0,2400,12000,B+,12,15,1,0,1,...,0,3,0,0,0,1,ЮЗАО,0,16425.314440,32
3,37,5.0,2400,12000,B+,4,15,1,0,1,...,0,3,0,0,0,1,ЮЗАО,0,16425.314440,32
4,39,5.0,2400,12000,B+,8,15,1,0,1,...,0,3,0,0,0,1,ЮЗАО,0,16425.314440,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11702,33524,16000.0,4167,66672000,B,12,16,0,1,0,...,9,303,703,0,0,1,САО,0,4215.373029,1281
11703,33574,25200.0,2917,73508400,B,1,0,0,1,0,...,10,27,34,0,0,1,ЮАО,0,8826.038745,112
11704,33577,13703.0,5473,75000000,B+,1,3,0,0,0,...,28,144,684,0,0,1,ЦАО,1,1147.612056,924
11705,33604,9700.0,8845,85796496,B,8,12,0,1,0,...,22,122,804,0,0,1,ЦАО,0,2646.298315,440


### Строим катбуст по 10 разным разбиениям, считаем средние метрики

In [6]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [7]:
# список фичей
features = [
    # В1: Помещение + здание, 
    'floor', 'maxfloor', 
    'isCU', 'isNonCU', 'isEE', 'isOfficeCenter', 'isMall',
    'inTTK', 'inGardenRing', 'AO',
    'space_m2',
    # В2: В1 + метро
    'minDistSubway',
    # В3: В2 + окружение
    'cafe_500', 'cafe_1000', 'cafe_2000',
    'fast_food_500', 'fast_food_1000', 'fast_food_2000', 
    'restaurant_500', 'restaurant_1000', 'restaurant_2000', 
    'bank_500', 'bank_1000', 'bank_2000', 
    'supermarket_500', 'supermarket_1000', 'supermarket_2000', 
    'clothes_500', 'clothes_1000', 'clothes_2000',
    'cosmetics_500', 'cosmetics_1000', 'cosmetics_2000',
    'electronics_500', 'electronics_1000', 'electronics_2000',
    'mall_500', 'mall_1000', 'mall_2000', 
    'histplaces_500', 'histplaces_1000', 'histplaces_2000',
    'distanceToCenter',
]
cat_features = ['isCU', 'isNonCU', 'isEE', 'inTTK', 'isOfficeCenter', 'isMall', 'AO']

In [8]:
mses = []
maes = []
absolutemaes = []
r2s = []

for seed in random_seeds:
    # Создаем генератор
    rng = np.random.default_rng(seed=seed)

    # берем 110 уникальных адресов
    test_index = rng.choice(max(dataset.address_index), size=300, replace=False)
    print(test_index[0:5])
    
    # делим датасет на тестовый и проверочный
    X_train = dataset.loc[~dataset['address_index'].isin(test_index)][features]
    y_train = dataset.loc[~dataset['address_index'].isin(test_index)]['target']
    X_test = dataset.loc[dataset['address_index'].isin(test_index)][features]
    y_test = dataset.loc[dataset['address_index'].isin(test_index)]['target']
    y_testRUR = dataset.loc[dataset['address_index'].isin(test_index)]['price_per_m2']
    print(y_train.shape, y_test.shape)
    
    # создаем модель
    model = CatBoostRegressor(
        iterations=400,
        learning_rate=0.015,
        random_state=seed,
        depth=10,
        loss_function='RMSE',
        cat_features = cat_features,
        verbose=0,
        l2_leaf_reg=2.0
    )
    
    # учим  модель
    model.fit(X_train, y_train)
    
    # получаем предикты
    predictions = model.predict(X_test)
    
    # MSE, MAE
    mses.append(mean_squared_error(y_test, predictions))
    maes.append(mean_absolute_error(y_test, predictions))
    
    # AbsoluteMAE
    absolutemaes.append(mean_absolute_error(np.exp(predictions), y_testRUR))
    
    # R2
    r2s.append(model.score(X_test, y_test))

[1340  956  222 1687  931]
(9850,) (1857,)
[ 721 1171  205  738  140]
(9945,) (1762,)
[ 527 1001  462 1460  333]
(9973,) (1734,)
[ 610  221 1051 1605  148]
(9314,) (2393,)
[ 305  179  154 1621  848]
(9944,) (1763,)
[ 903  691  699 1360 1281]
(9247,) (2460,)
[1049 1457 1625  914  371]
(9885,) (1822,)
[1207  496 1372 1513  545]
(9653,) (2054,)
[705 141 816 328 155]
(9176,) (2531,)
[724 606 465 627 808]
(9960,) (1747,)


In [9]:
print(f'#--- AVG MSE: {np.mean(mses):.5f}' )
print(f'#--- AVG MAE: {np.mean(maes):.5f}')
print(f'#ABS AVG MAE: {np.mean(absolutemaes):.5f}')
print(f'#--- AVG  R2: {np.mean(r2s):.5f}')

#--- AVG MSE: 0.13724
#--- AVG MAE: 0.27344
#ABS AVG MAE: 989.89818
#--- AVG  R2: 0.37020


In [10]:
### Хар-ки офиса + признаки локации
#--- AVG MSE: 0.16877
#--- AVG MAE: 0.31134
#ABS AVG MAE: 1108.95693
#--- AVG  R2: 0.22099
### -//- + метро
#--- AVG MSE: 0.16322
#--- AVG MAE: 0.29964
#ABS AVG MAE: 1074.62693
#--- AVG  R2: 0.25036
### -//- + окружение
#--- AVG MSE: 0.13724
#--- AVG MAE: 0.27344
#ABS AVG MAE: 989.89818
#--- AVG  R2: 0.37020

In [11]:
# Выводим значимость фичей
print("\nFeature Importances:")
dict(zip(features, model.get_feature_importance()))


Feature Importances:


{'floor': 2.0068618604167523,
 'maxfloor': 8.8629918533085,
 'isCU': 0.3538375455236957,
 'isNonCU': 0.05327499577778677,
 'isEE': 0.52067919271838,
 'isOfficeCenter': 0.5535446724438439,
 'isMall': 0.0,
 'inTTK': 0.0821749576074298,
 'inGardenRing': 0.21527886751186867,
 'AO': 3.26421492138115,
 'space_m2': 4.076991730249207,
 'minDistSubway': 6.433009909422375,
 'cafe_500': 5.9709637625947085,
 'cafe_1000': 3.2690440316623612,
 'cafe_2000': 1.6705052620978709,
 'fast_food_500': 2.0796359547390733,
 'fast_food_1000': 2.309256005430067,
 'fast_food_2000': 1.2665266079268984,
 'restaurant_500': 9.009766111640035,
 'restaurant_1000': 1.1509749472892594,
 'restaurant_2000': 2.795088775458479,
 'bank_500': 2.2240484833414143,
 'bank_1000': 1.6667424050921738,
 'bank_2000': 1.0297756201514163,
 'supermarket_500': 3.669180226149694,
 'supermarket_1000': 5.983604952236545,
 'supermarket_2000': 5.174908285129165,
 'clothes_500': 1.8706842206665542,
 'clothes_1000': 1.6864858431592384,
 'clothe

In [12]:
# Выводим топ-7 фичей по значимости для последнего разбиения
fi_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': model.get_feature_importance()
})
top_15_features = fi_df.sort_values(by='Importance', ascending=False).head(7)
display(top_15_features)

,Feature,Importance
18,restaurant_500,9.009766
1,maxfloor,8.862992
11,minDistSubway,6.433010
25,supermarket_1000,5.983605
12,cafe_500,5.970964
26,supermarket_2000,5.174908
10,space_m2,4.076992


### Строим линейную модель из sklearn

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [14]:
datasetLin = dataset
datasetLin.head()

,Unnamed: 0,space_m2,price_per_m2,total_price,office_class,floor,maxfloor,isCU,isNonCU,isEE,...,classACnt_500,classACnt_1000,classACnt_2000,isMoscowCity,isSkolkovo,inMKAD,AO,inGardenRing,distanceToCenter,address_index
0,32,5.0,2400,12000,B,9,16,1,0,1,...,0,3,0,0,0,1,ЮЗАО,0,16425.31444,32
1,34,5.0,2400,12000,B+,3,15,1,0,1,...,0,3,0,0,0,1,ЮЗАО,0,16425.31444,32
2,36,5.0,2400,12000,B+,12,15,1,0,1,...,0,3,0,0,0,1,ЮЗАО,0,16425.31444,32
3,37,5.0,2400,12000,B+,4,15,1,0,1,...,0,3,0,0,0,1,ЮЗАО,0,16425.31444,32
4,39,5.0,2400,12000,B+,8,15,1,0,1,...,0,3,0,0,0,1,ЮЗАО,0,16425.31444,32


In [15]:
datasetLin['nl_total_price'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['total_price'])]
datasetLin['nl_floor'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['floor'])]
datasetLin['nl_space_m2'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['space_m2'])]
datasetLin['nl_minDistSubway'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['minDistSubway'])]
datasetLin['nl_maxfloor'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['maxfloor'])]
env = [
    'cafe_500', 'cafe_1000', 'cafe_2000', 'fast_food_500', 'fast_food_1000', 'fast_food_2000', 'restaurant_500',
    'restaurant_1000', 'restaurant_2000', 'bank_500', 'bank_1000', 'bank_2000', 'supermarket_500', 'supermarket_1000', 'supermarket_2000', 'clothes_500', 'clothes_1000', 'clothes_2000',
    'cosmetics_500', 'cosmetics_1000', 'cosmetics_2000', 'electronics_500', 'electronics_1000', 'electronics_2000',
    'mall_500', 'mall_1000', 'mall_2000', 'histplaces_500', 'histplaces_1000', 'histplaces_2000', 'distanceToCenter'
]
for i in env:
    datasetLin['nl_' + i] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin[i])]
datasetLin = pd.get_dummies(datasetLin, columns=['office_class'], dtype=int)
datasetLin = pd.get_dummies(datasetLin, columns=['AO'], dtype=int)

linearFeatures = [
    'nl_floor', 'nl_maxfloor', 'isCU', 'isNonCU', 'isEE', 'isOfficeCenter', 'isMall',
    'inTTK', 'inGardenRing',
    'AO_CВАО', 'AO_ВАО', 'AO_ЗАО', 'AO_САО', 'AO_СЗАО', 'AO_ЮАО', 'AO_ЮВАО', 'AO_ЮЗАО',
    'nl_space_m2',
    # + метро
    'nl_minDistSubway',
    # + окружение
    'nl_cafe_500', 'nl_cafe_1000', 'nl_cafe_2000',
    'nl_fast_food_500', 'nl_fast_food_1000', 'nl_fast_food_2000', 
    'nl_restaurant_500', 'nl_restaurant_1000', 'nl_restaurant_2000', 
    'nl_bank_500', 'nl_bank_1000', 'nl_bank_2000', 
    'nl_supermarket_500', 'nl_supermarket_1000', 'nl_supermarket_2000',
    'nl_clothes_500', 'nl_clothes_1000', 'nl_clothes_2000',
    'nl_cosmetics_500', 'nl_cosmetics_1000', 'nl_cosmetics_2000',
    'nl_electronics_500', 'nl_electronics_1000', 'nl_electronics_2000',
    'nl_mall_500', 'nl_mall_1000', 'nl_mall_2000',
    'nl_histplaces_500', 'nl_histplaces_1000', 'nl_histplaces_2000',
    'nl_distanceToCenter'
]
exceptFeatures = []#['total_price', 'floor', 'minDistSubway', 'maxfloor'] + env
for i in features:
    if i not in linearFeatures + exceptFeatures:
        print(i)

floor
maxfloor
AO
space_m2
minDistSubway
cafe_500
cafe_1000
cafe_2000
fast_food_500
fast_food_1000
fast_food_2000
restaurant_500
restaurant_1000
restaurant_2000
bank_500
bank_1000
bank_2000
supermarket_500
supermarket_1000
supermarket_2000
clothes_500
clothes_1000
clothes_2000
cosmetics_500
cosmetics_1000
cosmetics_2000
electronics_500
electronics_1000
electronics_2000
mall_500
mall_1000
mall_2000
histplaces_500
histplaces_1000
histplaces_2000
distanceToCenter


In [16]:
mses = []
maes = []
absolutemaes = []
r2s = []

for seed in random_seeds:
    # Создаем генератор
    rng = np.random.default_rng(seed=seed)

    # берем 300 уникальных адресов
    test_index = rng.choice(max(datasetLin.address_index), size=300, replace=False)
    print(test_index[0:5])
    
    X_train = datasetLin.loc[~datasetLin['address_index'].isin(test_index)][linearFeatures]
    y_train = datasetLin.loc[~datasetLin['address_index'].isin(test_index)]['target']
    X_test = datasetLin.loc[datasetLin['address_index'].isin(test_index)][linearFeatures]
    y_test = datasetLin.loc[datasetLin['address_index'].isin(test_index)]['target']
    y_testRUR = dataset.loc[datasetLin['address_index'].isin(test_index)]['price_per_m2']
    
    # учим модель
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    # считаем метрики
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    absolutemae = mean_absolute_error(np.exp(y_pred), y_testRUR)
    
    mses.append(mse)
    r2s.append(r2)
    maes.append(mae)
    absolutemaes.append(absolutemae)

[1340  956  222 1687  931]
[ 721 1171  205  738  140]
[ 527 1001  462 1460  333]
[ 610  221 1051 1605  148]
[ 305  179  154 1621  848]
[ 903  691  699 1360 1281]
[1049 1457 1625  914  371]
[1207  496 1372 1513  545]
[705 141 816 328 155]
[724 606 465 627 808]


In [17]:
print(f'#--- AVG MSE: {np.mean(mses):.5f}' )
print(f'#--- AVG MAE: {np.mean(maes):.5f}')
print(f'#ABS AVG MAE: {np.mean(absolutemaes):.5f}')
print(f'#--- AVG  R2: {np.mean(r2s):.5f}')

#--- AVG MSE: 0.14899
#--- AVG MAE: 0.28798
#ABS AVG MAE: 1034.22188
#--- AVG  R2: 0.31240


In [18]:
### Хар-ки офиса + признаки локации
#--- AVG MSE: 0.17031
#--- AVG MAE: 0.31588
#ABS AVG MAE: 1115.88274
#--- AVG  R2: 0.21198
### -//- + метро
#--- AVG MSE: 0.15530
#--- AVG MAE: 0.29788
#ABS AVG MAE: 1056.68605
#--- AVG  R2: 0.27914
### -//- + окружение
#--- AVG MSE: 0.14899
#--- AVG MAE: 0.28798
#ABS AVG MAE: 1034.22188
#--- AVG  R2: 0.31240

In [19]:
# коэфтцтенты
coefficients = model.coef_
intercept = model.intercept_

print(f"Intercept: {intercept}")
for i in range(len(coefficients)):
    if abs(coefficients[i]) > 0.12:
        print(round(coefficients[i], 5), linearFeatures[i])

Intercept: 7.834517418841978
-0.18691 inTTK
-0.15807 AO_ВАО
0.17585 nl_cafe_500
0.18786 nl_cafe_2000
-0.18783 nl_fast_food_2000
0.14958 nl_restaurant_2000
-0.14178 nl_supermarket_2000
0.13769 nl_mall_1000


In [20]:
coef_df = pd.DataFrame(
    {
        "Feature": X_train.columns,
        "Coefficient": model.coef_,  # Use model.coef_[0] for logistic regression
    }
)

# 2. Sort by absolute value to find the strongest positive and negative impacts
coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()
top = coef_df.sort_values(by="Abs_Coefficient", ascending=False)

# 3. Display the result
display(top[["Feature", "Coefficient"]])

,Feature,Coefficient
21,nl_cafe_2000,0.187856
24,nl_fast_food_2000,-0.187835
7,inTTK,-0.186912
19,nl_cafe_500,0.175849
10,AO_ВАО,-0.158072
27,nl_restaurant_2000,0.149578
33,nl_supermarket_2000,-0.141778
44,nl_mall_1000,0.137688
13,AO_СЗАО,-0.116586
9,AO_CВАО,0.110733


In [21]:
tempdf = pd.DataFrame()
for i in ['nl_minDistSubway', 'nl_restaurant_500', 'nl_restaurant_1000', 'nl_restaurant_2000',
         'nl_supermarket_500', 'nl_supermarket_1000', 'nl_supermarket_2000',
         'nl_fast_food_500', 'nl_fast_food_1000', 'nl_fast_food_2000',
         'nl_cafe_500', 'nl_cafe_1000', 'nl_cafe_2000',
         'nl_histplaces_500', 'nl_histplaces_1000', 'nl_histplaces_2000']:
    t = top.loc[top['Feature'] == i].reset_index(drop=True)
    tempdf = pd.concat([tempdf, t])
tempdf[['Feature', 'Coefficient']]
print(tempdf[['Coefficient']].to_string(index=False))

 Coefficient
   -0.107062
    0.060460
   -0.010412
    0.149578
   -0.018318
   -0.079038
   -0.141778
   -0.042021
   -0.101489
   -0.187835
    0.175849
    0.109149
    0.187856
   -0.021744
    0.052132
    0.037513
